In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("hospitals93to98.csv")

# Preview
df.head()

,IcdChapter,Field,FY1993,FY1994,FY1995,FY1996,FY1997,FY1998
0,0. Not Reported,PatientDays,"257,965","55,582","128,507","182,226","61,599","685,879"
1,0. Not Reported,Separations,"37,178","6,146","3,832","4,861","1,558","53,575"
2,1. Infectious and Parasitic Diseases,PatientDays,"311,221","313,386","324,693","311,560","306,688","1,567,548"
3,1. Infectious and Parasitic Diseases,Separations,"75,857","78,323","84,631","80,864","79,148","398,823"
4,2. Neoplasms,PatientDays,"1,686,919","1,707,437","1,795,751","1,770,559","1,777,452","8,738,118"


In [2]:
# Check missing values
print(df.isnull().sum())

# Fill missing values (simple strategy)
df = df.fillna("")

IcdChapter    0
Field         0
FY1993        0
FY1994        0
FY1995        0
FY1996        0
FY1997        0
FY1998        0
dtype: int64


In [4]:
print(df.dtypes)

IcdChapter    str
Field         str
FY1993        str
FY1994        str
FY1995        str
FY1996        str
FY1997        str
FY1998        str
dtype: object


In [5]:
df["combined_text"] = df[text_cols].astype(str).agg(" ".join, axis=1)

df["combined_text"].head()

0    0. Not Reported PatientDays 257,965 55,582 128...
1    0. Not Reported Separations 37,178 6,146 3,832...
2    1. Infectious and Parasitic Diseases PatientDa...
3    1. Infectious and Parasitic Diseases Separatio...
4    2. Neoplasms PatientDays 1,686,919 1,707,437 1...
Name: combined_text, dtype: str

In [6]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df["combined_text"].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=500)
X_text = vectorizer.fit_transform(df["clean_text"])

In [8]:
numeric_df = df.select_dtypes(include=[np.number]).fillna(0)

# Combine text + numeric
from scipy.sparse import hstack

X = hstack([X_text, numeric_df.values])

In [9]:
from sklearn.cluster import KMeans

k = 5  # you can tune this
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

df["cluster"] = kmeans.fit_predict(X)

df[["cluster"]].head()

,cluster
0,4
1,4
2,2
3,2
4,4


In [10]:
df.to_csv("hospitals_clustered_output.csv", index=False)